# Comparación con NLP: spaCy y Transformers

**Responsable:** Persona 1

Objetivo: analizar el set de prueba con dos herramientas de NLP para español y compararlas con
nuestra gramática libre de contexto.

| Herramienta | Salida |
| --- | --- |
| spaCy `es_core_news_sm` | `results/spacy.jsonl` |
| Modelo de HuggingFace Transformers para español | `results/transformers.jsonl` |

Las dos salidas usan el formato común de `evaluation/formats.py` (ver `docs/formato_salida.md`),
el mismo que usan el parser y el LLM, para que la Persona 4 calcule las métricas de todos igual.

**Antes de correrlo:**

```bash
pip install -r requirements-experiments.txt
python -m spacy download es_core_news_sm
```

Preguntas que este notebook debe responder (van al informe):

1. ¿En qué oraciones spaCy y Transformers analizan bien lo que la CFG **no puede**? Pista: las oraciones con `gramatical = true` y `cubierta = false`.
2. ¿En qué oraciones la CFG acierta y ellos **no**? Pista: errores de concordancia, que una herramienta estadística no rechaza.

In [ ]:
# Configuración: el notebook vive en notebooks/, el código del proyecto en la raíz.
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from evaluation.formats import Prediction, load_test_set, save_predictions, results_path

items = load_test_set()
print(f"{len(items)} oraciones en el set de prueba")

In [ ]:
# Vista rápida del set de prueba.
for item in items:
    marca = "✓" if item.covered else "✗"
    print(f"{item.id}  {item.level:<10} {marca}  {item.sentence}")

## 1. spaCy

spaCy no "rechaza" oraciones: siempre devuelve un análisis (categorías, rasgos y dependencias).
Para comparar la aceptación hay que definir un criterio, por ejemplo revisar la concordancia con
los rasgos morfológicos que da spaCy (`token.morph`).

In [ ]:
import spacy

try:
    nlp = spacy.load("es_core_news_sm")
except OSError:
    raise SystemExit("Falta el modelo: python -m spacy download es_core_news_sm")

In [ ]:
from experiments.tagsets import upos_to_ours


def spacy_tokens(doc) -> list[dict]:
    """Análisis de spaCy palabra por palabra (sin la puntuación)."""
    rows = []
    for token in doc:
        category = upos_to_ours(token.pos_)
        if category is None:
            continue
        rows.append({
            "palabra": token.text,
            "upos": token.pos_,
            "categoria": category,
            "rasgos": str(token.morph),
            "dep": token.dep_,
            "cabeza": token.head.text,
        })
    return rows


doc = nlp(items[0].sentence)
spacy_tokens(doc)

In [ ]:
def spacy_decision(doc) -> tuple[bool, str | None, str]:
    """¿Acepta spaCy la oración? Devuelve (aceptada, tipo_error, detalle).

    TODO(P1): definir el criterio. Propuesta:
      - concordancia DET/ADJ con su sustantivo: tokens con dep "det" o "amod"
        comparan Gender y Number con token.head;
      - sujeto-verbo: el token con dep "nsubj" compara Number (y Person) con su head;
      - si no hay ningún verbo (pos VERB/AUX), rechazar como "sintactico".
    """
    raise NotImplementedError("TODO(P1): spacy_decision")

In [ ]:
spacy_predictions = []
for item in items:
    doc = nlp(item.sentence)
    tokens = spacy_tokens(doc)
    accepted, error_type, detail = spacy_decision(doc)
    spacy_predictions.append(Prediction(
        id=item.id,
        system="spacy",
        accepted=accepted,
        tree=None,  # spaCy da dependencias, no nuestro árbol de constituyentes
        categories=[t["categoria"] for t in tokens],
        error_type=error_type,
        detail=detail,
        extra={"tokens": tokens},
    ))

print("Guardado en", save_predictions("spacy", spacy_predictions))

## 2. HuggingFace Transformers

TODO(P1): elegir el modelo. Tiene que ser de **clasificación de tokens** (POS tagging) para español.
Buscar en huggingface.co, por ejemplo con el filtro *token-classification* e idioma *es*, y
anotar aquí qué modelo se usó, su tamaño y con qué etiquetas trabaja (UPOS u otras).

La primera vez que se carga, el modelo se descarga (puede pesar varios cientos de MB).

In [ ]:
from transformers import pipeline

MODEL = "TODO(P1): nombre/del-modelo"   # p. ej. un modelo de POS tagging en español

tagger = pipeline("token-classification", model=MODEL, aggregation_strategy="simple")

In [ ]:
def transformers_tokens(sentence: str) -> list[dict]:
    """Etiqueta cada palabra. TODO(P1): revisar el nombre del campo de la etiqueta
    ("entity_group" con aggregation_strategy="simple") y si el tagset es UPOS;
    si no lo es, agregar su tabla de conversión en experiments/tagsets.py."""
    rows = []
    for entity in tagger(sentence):
        tag = entity["entity_group"]
        rows.append({"palabra": entity["word"], "etiqueta": tag, "categoria": upos_to_ours(tag)})
    return rows


transformers_tokens(items[0].sentence)

In [ ]:
def transformers_decision(sentence: str, tokens: list[dict]) -> tuple[bool, str | None, str]:
    """¿Acepta el modelo la oración? TODO(P1): un etiquetador tampoco rechaza.
    Opciones: aceptar siempre (y comparar solo categorías), o usar los rasgos si
    el modelo los da, o un criterio de orden (DET antes de SUST, debe haber un V)."""
    raise NotImplementedError("TODO(P1): transformers_decision")


transformers_predictions = []
for item in items:
    tokens = transformers_tokens(item.sentence)
    accepted, error_type, detail = transformers_decision(item.sentence, tokens)
    transformers_predictions.append(Prediction(
        id=item.id,
        system="transformers",
        accepted=accepted,
        categories=[t["categoria"] for t in tokens if t["categoria"]],
        error_type=error_type,
        detail=detail,
        extra={"tokens": tokens, "modelo": MODEL},
    ))

print("Guardado en", save_predictions("transformers", transformers_predictions))

## 3. Comparación con la CFG

Necesita `results/parser.jsonl` (Persona 2: `python -m evaluation.run_parser`).

In [ ]:
from evaluation.formats import load_predictions

systems = [s for s in ("parser", "spacy", "transformers") if results_path(s).exists()]
by_system = {s: {p.id: p for p in load_predictions(s)} for s in systems}

print(f"{'id':<6} {'gram':<5} {'cub':<5} " + " ".join(f"{s:<13}" for s in systems))
for item in items:
    cells = []
    for s in systems:
        p = by_system[s].get(item.id)
        cells.append("—" if p is None else ("acepta" if p.accepted else f"rechaza:{p.error_type}"))
    print(f"{item.id:<6} {str(item.grammatical):<5} {str(item.covered):<5} " + " ".join(f"{c:<13}" for c in cells))

### Conclusiones (para el informe)

TODO(P1):

- Oraciones que spaCy/Transformers analizan bien y la CFG no puede (¿por qué? orden libre, palabras fuera del léxico, construcciones no previstas...).
- Oraciones que la CFG acierta y ellos no (¿concordancia?).
- Diferencia de fondo: reglas explícitas frente a modelos estadísticos.